# Vocab vs Fixed-Topic Cosine Similarity

This notebook loads the OPT-2.7b embedding matrix and vocabulary from
`data/complete/`, then computes the **cosine similarity** between **every
vocab token** and a small set of **fixed topic words** you provide.

**Steps before running:**
1. Upload the `data/complete/` folder (`opt27_embeddings.npy` + `opt27_vocab.csv`)
   to your Google Drive.
2. Edit `DATA_DIR` below so it points at that folder.
3. Edit the `TOPICS` list in the "Fixed Topics" cell with your own words.

In [2]:
import os
from collections import defaultdict

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive",force_remount=True)

Mounted at /content/drive


In [3]:

DATA_DIR = "/content/drive/MyDrive/minor_project/"

embeddings = np.load(os.path.join(DATA_DIR, "opt27_embeddings.npy")).astype(np.float32)
vocab_df   = pd.read_csv(os.path.join(DATA_DIR, "opt27_vocab.csv"))

print("embeddings:", embeddings.shape, embeddings.dtype)
print("vocab rows:", len(vocab_df))


vocab_df = vocab_df[vocab_df["token_id"].isin(range(embeddings.shape[0]))]
vocab_df = vocab_df.dropna(subset=["token"])
token_ids = vocab_df["token_id"].to_numpy()
tokens    = vocab_df["token"].to_numpy()
emb       = embeddings[token_ids]

print("aligned vocab x embedding:", emb.shape)

embeddings: (50272, 2560) float32
vocab rows: 50265
aligned vocab x embedding: (50260, 2560)


In [4]:

id_to_token = dict(zip(token_ids, tokens))
token_to_id = {tok: tid for tid, tok in id_to_token.items()}

id_to_index = {tid: i for i, tid in enumerate(token_ids)}

# OPT BPE prepends a "Ġ" space-marker to most word tokens.
# Map a cleaned (lowercased, no "Ġ") form to every matching token_id.
clean_to_ids = defaultdict(list)
for tid, tok in id_to_token.items():
    clean_to_ids[tok.replace("Ġ", "").lower()].append(tid)

def resolve_topic(word):
    """Return token_id(s) for a topic word.

    Tries the plain word, then the 'Ġ'-prefixed BPE form, then falls back
    to a cleaned-text lookup (handles casing and the 'Ġ' marker).
    """
    for cand in (word, "Ġ" + word):
        if cand in token_to_id:
            return [token_to_id[cand]]
    return clean_to_ids.get(word.lower(), [])

# Unit-normalize each embedding row once (rows already aligned)
norm_emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)

## Fixed Topics

Edit the `TOPICS` list below with your own topic words. Every topic should
be a word/phrase that exists in the model vocab (e.g. `"technology"` maps to
the token `Ġtechnology`). The next cell reports which token each topic
resolved to, so you can verify.

In [5]:
TOPICS = [
    "technology",
    "medicine",
    "sports",
    "politics",
    "science",
    "entertainment",
    "finance",
    "history"
]

In [6]:
# ------------------------------------------------------------------
# Resolve each topic to its vocab token_id(s)
# ------------------------------------------------------------------
topic_ids = []
for word in TOPICS:
    ids = resolve_topic(word)
    resolved = [id_to_token[i] for i in ids]
    print(f"{word!r:20} -> {resolved}")
    topic_ids.append(ids)

'technology'         -> ['technology']
'medicine'           -> ['Ġmedicine']
'sports'             -> ['sports']
'politics'           -> ['politics']
'science'            -> ['science']
'entertainment'      -> ['Ġentertainment']
'finance'            -> ['Ġfinance']
'history'            -> ['history']


In [7]:
# ------------------------------------------------------------------
# Cosine similarity matrix, shape (num_vocab, num_topics):
#   sim[i, j] = cosine(embedding(vocab_token_i), embedding(topic_j))
#
# Each topic is represented by the mean of its resolved token embeddings
# (re-normalized to unit length).
# ------------------------------------------------------------------
sim = np.zeros((len(token_ids), len(TOPICS)), dtype=np.float32)

for j, ids in enumerate(topic_ids):
    if not ids:
        print(f"WARNING: topic {TOPICS[j]!r} not found in vocab, skipping")
        continue
    rows = [id_to_index[i] for i in ids]
    topic_vec = norm_emb[rows].mean(axis=0)
    topic_vec = topic_vec / np.linalg.norm(topic_vec)
    sim[:, j] = norm_emb @ topic_vec

print("similarity matrix:", sim.shape)

similarity matrix: (50260, 8)


In [8]:
# ------------------------------------------------------------------
# Top-N most similar vocab tokens for each topic
# ------------------------------------------------------------------
TOP_N = 25

for j, word in enumerate(TOPICS):
    order = np.argsort(-sim[:, j])
    top = pd.DataFrame({
        "token":      tokens[order[:TOP_N]],
        "similarity": sim[order[:TOP_N], j].round(4),
    }).reset_index(drop=True)
    print(f"\n=== Most similar to '{word}' ===")
    display(top)


=== Most similar to 'technology' ===


,token,similarity
0,technology,1.0000
1,Technology,0.7134
2,ĠTechnology,0.6306
3,Ġtechnology,0.6282
4,Ġtechnologies,0.5620
5,Ġtechnological,0.5091
6,ĠTECH,0.5017
7,Ġtechnologically,0.4609
8,tech,0.4376
9,otechnology,0.4127



=== Most similar to 'medicine' ===


,token,similarity
0,Ġmedicine,1.0000
1,ĠMedicine,0.6859
2,Ġmedicines,0.5869
3,Ġmedication,0.4981
4,Medic,0.3821
5,Ġmedications,0.3794
6,Ġmedical,0.3786
7,Ġmedicinal,0.3659
8,Ġmedic,0.3222
9,Ġmed,0.3171



=== Most similar to 'sports' ===


...

,token,similarity
0,Ġentertainment,1.0000
1,ĠEntertainment,0.5993
2,Ġentertain,0.5287
3,Ġentertaining,0.4505
4,Ġamusement,0.4135
5,Ġentertained,0.4110
6,Ġentert,0.3578
7,Ġmusic,0.3102
8,tainment,0.3023
9,Ġenjoyment,0.3000



=== Most similar to 'finance' ===


,token,similarity
0,Ġfinance,1.0000
1,ĠFinance,0.7320
2,Ġfinancing,0.4813
3,Ġfinances,0.4648
4,Ġfinancial,0.4316
5,Ġfinan,0.4184
6,inance,0.3970
7,Ġfinanced,0.3689
8,financial,0.3491
9,Financial,0.3231



=== Most similar to 'history' ===


,token,similarity
0,history,1.0000
1,History,0.7101
2,ĠHistory,0.6912
3,Ġhistory,0.6735
4,Ġhistories,0.6128
5,ISTORY,0.5373
6,Ġhistorians,0.4829
7,hist,0.4516
8,Ġhistor,0.4485
9,Ġhistorian,0.4348


In [9]:

topicality = pd.DataFrame({
    "token":          tokens,
    "topic":          [TOPICS[j] for j in sim.argmax(axis=1)],
    "max_similarity": sim.max(axis=1).round(4),
}).sort_values("max_similarity", ascending=False).reset_index(drop=True)

display(topicality.head(50))


out = pd.DataFrame({"token_id": token_ids, "token": tokens})
for j, word in enumerate(TOPICS):
    out[f"sim_{word}"] = sim[:, j].round(4)
out.to_csv(os.path.join(DATA_DIR, "topic_cosine_similarity.csv"), index=False)
print("\nsaved topic_cosine_similarity.csv")

,token,topic,max_similarity
0,Ġfinance,finance,1.0000
1,Ġentertainment,entertainment,1.0000
2,Ġmedicine,medicine,1.0000
3,science,science,1.0000
4,technology,technology,1.0000
5,sports,sports,1.0000
6,history,history,1.0000
7,politics,politics,1.0000
8,ĠFinance,finance,0.7320
9,Technology,technology,0.7134



saved topic_cosine_similarity.csv
